# Backtest SMACross

Run the SMACross strategy on a simulated exchange venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + SMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.sma_cross import SMACross, SMACrossConfig

from charts import plot_sma_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC" 
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 1                 # Backtesting leverage (margin_init = 1/1 = 100%)
SAVE_TEARSHEET   = False

# Fee overrides
MAKER_FEE = Decimal("0.000200")  # 0.02%
TAKER_FEE = Decimal("0.000500")  # 0.05%

# SMA params
FAST_SMA = 15
SLOW_SMA = 25

# Sweep grids
FAST_PERIODS = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100] + [FAST_SMA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_SMA]))

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"SMACross_{INSTRUMENT_ID}_{BAR_INTERVAL}_f{FAST_SMA}_s{SLOW_SMA}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(
    instrument, 
    LEVERAGE,
    MAKER_FEE,
    TAKER_FEE
)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add SMACross strategy + run ───────────────────────────
config = SMACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    fast_sma_period=FAST_SMA,
    slow_sma_period=SLOW_SMA,
)
strategy = SMACross(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

import numpy as np
import pandas as pd

# Get daily account balances from the account report
balances = account_report['total'].astype(float)  # or whatever column has equity

# Compute daily returns
returns = balances.pct_change().dropna()

# Annualized Sharpe (assuming risk-free = 0, 252 trading days)
sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
print(f"Manual Sharpe: {sharpe:.4f}")
print(f"Manual vol (annualized): {returns.std() * np.sqrt(252):.4f}")
print(f"Number of return observations: {len(returns)}")

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"SMACross({FAST_SMA}/{SLOW_SMA}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Price chart with SMA overlay + trade markers ──────────

fig = plot_sma_cross(
    bars, fills_report,
    fast_period=FAST_SMA,
    slow_period=SLOW_SMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────
plot_equity_curve(analyzer, account_report, f"SMACross({FAST_SMA}/{SLOW_SMA})  {INSTRUMENT_ID} {BAR_INTERVAL}")

In [ ]:
# ── Cell 10: Summary statistics ────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep ──────────────────────────────────────
#
# Grid search over fast/slow SMA period combinations.

def sma_factory(eng, params):
    cfg = SMACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        fast_sma_period=params["fast"],
        slow_sma_period=params["slow"],
    )
    eng.add_strategy(SMACross(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=sma_factory,
    strategy_name="SMACross",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap ──────────────────────────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label="Slow SMA Period", col_label="Fast SMA Period",
    title=f"Total PnL ({CURRENCY}) by SMA Parameters",
)

In [ ]:
# ── Cell 13: TradingView Interactive Backtest ──────────────────────

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_SMA,
    slow_period=SLOW_SMA,
    ma_type="SMA",
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
)
# Then just open the file in your browser

In [ ]:
# ── Cell 14: Save notebook snapshot ────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_sma_cross.ipynb", RESULT_NAME)
#save_notebook_html("backtest_sma_cross.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 15: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")